# Compare the current code against a saved baseline

Baselines are produced by `scripts/baselines/make_baseline.py` and stored in `scripts/baselines/data/`.
Each one holds, for every reference case, the optimal slack $s$ of each Monte Carlo sample
($s < 0$: the assembly fails), computed with a fixed seed. Recomputing the same cases with the
current code shows, case by case, how far a change moves the results:

- **Pf** before and after,
- **reclassified** samples (assembly verdict flipped),
- **max |Δs|**, the largest slack change on a sample,
- **nI**, the number of interface (contact) constraints, which reveals lost or added contacts,
- **solver failures** (`s = NaN`), e.g. unbounded problems when contacts are missing.

Cases built from hand-coded matrices (`model3`, `model4`) are controls: changes to the automatic
pipeline must leave them untouched. Runtime: about 2 minutes on a laptop (scipy MILP, no GPU).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "Baselines" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts" / "baselines"))
import baseline_cases as bc

REFERENCE_TAG = "pre-fix"
sorted(p.stem for p in bc.DATA_DIR.glob("*.json"))

In [ ]:
reference = bc.load(REFERENCE_TAG)
meta, ref_slacks = reference
print("baseline", meta["tag"], "| code revision", meta["git_revision"], "| created", meta["created"])
cases = [c for c in bc.CASES if c.name in ref_slacks]
current = bc.run(cases)

In [ ]:
table = bc.compare(reference, current)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.4g}".format)
table

## Failure probability per case

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(table))
ax.bar(x - 0.2, table["Pf_ref"], width=0.4, label=f"baseline ({REFERENCE_TAG})")
ax.bar(x + 0.2, table["Pf_now"], width=0.4, label="current code")
ax.set_xticks(x, table.index, rotation=45, ha="right")
ax.set_ylabel("Pf")
ax.legend()
fig.tight_layout()

## Slack per sample: baseline vs current

Points on the diagonal are unchanged. Points in the off-diagonal quadrants (red) changed verdict.

In [ ]:
changed = [name for name in table.index if not (table.loc[name, "max_abs_ds"] < 1e-9)]
ncol = 3
nrow = max(1, int(np.ceil(len(changed) / ncol)))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow), squeeze=False)
for ax, name in zip(axes.flat, changed):
    s_ref, s_now = ref_slacks[name], current[name]["slack"]
    flipped = (s_ref < 0) != (s_now < 0)
    ax.scatter(s_ref[~flipped], s_now[~flipped], s=4, alpha=0.4)
    ax.scatter(s_ref[flipped], s_now[flipped], s=8, color="red")
    lim = np.nanmax(np.abs(np.concatenate([s_ref, s_now]))) if np.isfinite(np.concatenate([s_ref, s_now])).any() else 1
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set_title(f"{name}\n{flipped.sum()} reclassified", fontsize=9)
    ax.set_xlabel("s baseline")
    ax.set_ylabel("s current")
for ax in axes.flat[len(changed):]:
    ax.axis("off")
fig.tight_layout()
print("unchanged cases:", [n for n in table.index if n not in changed])

## Saving the current results as a new baseline

Once a change is understood and accepted, record it so the next change is measured against it:

```python
bc.save(current, "post-overlap-fix", cases)
```